# Lekcija 02 - Raziskovanje Microsoft Agent Frameworka

**Microsoft Agent Framework (MAF)** je enotno ogrodje za gradnjo AI agentov. Ponuja čisto, sestavljivo arhitekturo s štirimi osnovnimi gradniki:

- **Client** – poveže se z AI model endpointom in upravlja komunikacijo
- **Agent** – ovije klienta z navodili in definicijami orodij
- **Tools** – razširijo zmogljivosti agenta z lastnimi funkcijami, ki jih lahko model pokliče
- **Session** – ohranja zgodovino pogovora za interakcije z več potezami

V tej lekciji bomo zgradili **agenta za rezervacijo potovanj**, ki preverja razpoložljivost destinacij z uporabo teh konceptov.


## Namestitev


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Razumevanje arhitekture Agent Frameworka

Microsoft Agent Framework sledi plastni arhitekturi:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Odjemalec** – `AzureAIProjectAgentProvider` se poveže z Azure OpenAI namestitvijo. Upravlja avtentikacijo, oblikovanje zahtevka in razčlenjevanje odgovorov.
2. **Agent** – Ustvari se iz odjemalca prek `provider.create_agent()`, agent združuje dostop do modela z navodili (sistemski poziv) in orodji.
3. **Orodja** – Python funkcije, okrašene z `@tool`, ki jih lahko agent pokliče za izvajanje dejanj ali pridobivanje podatkov.
4. **Seja** – Objekt `AgentSession` (ustvarjen prek `agent.create_session()`), ki shranjuje zgodovino pogovora in omogoča večkratni dialog, kjer se agent spomni prejšnjega konteksta.

Zgradimo vsako plast korak za korakom.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Dodajanje orodij z dekoratorjem @tool

Orodja omogočajo agentom izvajanje dejanj, ki presegajo generiranje besedila. Dekorator `@tool` pretvori običajno funkcijo Pythona v nekaj, kar lahko agent pokliče.

Ključne točke:
- Uporabite `Annotated[type, "opis"]`, da model razume vsak parameter.
- Dokumentacijska vrstica postane opis orodja, ki ga model vidi.
- `approval_mode="never_require"` pomeni, da se orodje samodejno zažene brez potrditve uporabnika.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Ustvarjanje agenta z orodji

Zdaj združimo odjemalca, navodila in orodja v agenta. `instructions` delujejo kot sistemski poziv — opredeljujejo osebnost in vedenje agenta.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Večkratna pogajanja s sejami

`AgentSession` (ustvarjen preko `agent.create_session()`) sledi vsem sporočilom v pogovoru. Z posredovanjem iste seje v vsak klic `agent.run()` ima agent dostop do celotne zgodovine pogovora in se lahko sklicuje na prejšnja sporočila.

Posredujemo `tools=[check_destination_availability]`, da lahko agent med vsakim korakom pokliče našega preverjevalca razpoložljivosti.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Povzetek

V tej lekciji ste raziskali štiri stebre Microsoft Agent Framework:

| Koncept | Kaj ste se naučili |
|---------|---------------------|
| **Stranka** | `AzureAIProjectAgentProvider` se poveže z Azure OpenAI z avtentikacijo na podlagi poverilnic |
| **Agent** | `provider.create_agent()` poveže model s navodili in imenom |
| **Orodja** | Dekorator `@tool` omogoča agentu klicanje Python funkcij |
| **Seja** | `agent.create_session()` ohranja zgodovino pogovora skozi več zamenjav |

Ti gradniki skupaj sestavijo agente, ki lahko vodijo naravne pogovore, kličejo zunanje funkcije in ohranjajo kontekst — temelj za bolj napredne agente v poznejših lekcijah.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Opozorilo**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku velja za verodostojen vir. Za pomembne informacije priporočamo strokovni človeški prevod. Nismo odgovorni za morebitna nesporazumevanja ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
